In [0]:
from pyspark.sql import functions as F

CATALOGO = "mvp"
ESQUEMA = "staging"

TABELA_SILVER = f"{CATALOGO}.{ESQUEMA}.silver_diabetes_clean"
TABELA_GOLD   = f"{CATALOGO}.{ESQUEMA}.gold_fato_diabetes"

# 1. Leitura da Camada Silver
df_silver = spark.table(TABELA_SILVER)

# 2. Categorizações e Regras de Negócio (Camada Gold)
df_gold = df_silver \
    .withColumn("faixa_etaria", 
        F.when(F.col("age") < 30, "Jovem (<30)")
         .when((F.col("age") >= 30) & (F.col("age") <= 59), "Adulto (30-59)")
         .otherwise("Idoso (60+)")
    ) \
    .withColumn("categoria_bmi", 
        F.when(F.col("bmi") < 25.0, "1. Normal")
         .when((F.col("bmi") >= 25.0) & (F.col("bmi") < 30.0), "2. Sobrepeso")
         .otherwise("3. Obesidade")
    ) \
    .withColumn("categoria_glicose", 
        F.when(F.col("fasting_blood_sugar") < 100.0, "1. Normal")
         .when((F.col("fasting_blood_sugar") >= 100.0) & (F.col("fasting_blood_sugar") <= 125.0), "2. Alterada")
         .otherwise("3. Elevada")
    )

# 3. Escrita na Camada Gold em Delta Lake
df_gold.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(TABELA_GOLD)

print(f"✓ Camada Gold criada com sucesso! Linhas: {spark.table(TABELA_GOLD).count()}")

In [0]:
%sql
-- Fixar o catálogo e o esquema de trabalho
USE CATALOG mvp;
USE SCHEMA staging;

-- =========================================================================
-- 1. DIMENSÃO SECUNDÁRIA (Sub-dimensão normalizada para o Floco de Neve)
-- =========================================================================
CREATE OR REPLACE TABLE mvp.staging.dim_categoria_imc AS
SELECT 
    ROW_NUMBER() OVER (ORDER BY categoria_bmi) AS id_categoria_bmi,
    categoria_bmi AS descricao_categoria,
    CASE 
        WHEN categoria_bmi = '1. Normal' THEN 'Risco Baixo'
        WHEN categoria_bmi = '2. Sobrepeso' THEN 'Risco Moderado'
        ELSE 'Risco Elevado'
    END AS classificacao_risco_metabolico
FROM (
    SELECT DISTINCT categoria_bmi 
    FROM mvp.staging.gold_fato_diabetes
);

-- =========================================================================
-- 2. DIMENSÃO PRIMÁRIA (Pacientes - Conectada à sub-dimensão IMC)
-- =========================================================================
CREATE OR REPLACE TABLE mvp.staging.dim_paciente AS
SELECT 
    MONOTONICALLY_INCREASING_ID() AS id_paciente,
    g.age,
    g.faixa_etaria,
    g.family_history_diabetes,
    g.physical_activity_level,
    c.id_categoria_bmi
FROM mvp.staging.gold_fato_diabetes g
LEFT JOIN mvp.staging.dim_categoria_imc c 
       ON g.categoria_bmi = c.descricao_categoria;

-- =========================================================================
-- 3. DIMENSÃO PRIMÁRIA (Categorias de Glicemia)
-- =========================================================================
CREATE OR REPLACE TABLE mvp.staging.dim_glicemia AS
SELECT 
    ROW_NUMBER() OVER (ORDER BY categoria_glicose) AS id_glicemia,
    categoria_glicose AS faixa_glicemica
FROM (
    SELECT DISTINCT categoria_glicose 
    FROM mvp.staging.gold_fato_diabetes
);

-- =========================================================================
-- 4. TABELA FATO CENTRAL (Contém Métricas e Chaves Estrangeiras)
-- =========================================================================
CREATE OR REPLACE TABLE mvp.staging.fato_diabetes_snowflake AS
SELECT 
    p.id_paciente,
    gl.id_glicemia,
    g.bmi,
    g.fasting_blood_sugar,
    g.diabetes AS flag_diabetes
FROM mvp.staging.gold_fato_diabetes g
INNER JOIN mvp.staging.dim_paciente p 
        ON g.age = p.age 
       AND g.family_history_diabetes = p.family_history_diabetes 
       AND g.physical_activity_level = p.physical_activity_level
INNER JOIN mvp.staging.dim_glicemia gl 
        ON g.categoria_glicose = gl.faixa_glicemica;

In [0]:
%sql
SELECT 
    p.faixa_etaria,
    c.descricao_categoria AS categoria_imc,
    c.classificacao_risco_metabolico,
    gl.faixa_glicemica,
    COUNT(f.id_paciente) AS total_pacientes,
    SUM(f.flag_diabetes) AS casos_diabetes,
    ROUND(AVG(f.flag_diabetes) * 100, 2) AS pct_prevalencia
FROM mvp.staging.fato_diabetes_snowflake f
-- Conexão com Dimensão Primária (Paciente)
JOIN mvp.staging.dim_paciente p 
  ON f.id_paciente = p.id_paciente
-- Conexão da Dimensão com Sub-dimensão (Característica do Snowflake)
JOIN mvp.staging.dim_categoria_imc c 
  ON p.id_categoria_bmi = c.id_categoria_bmi
-- Conexão com a outra Dimensão Primária (Glicemia)
JOIN mvp.staging.dim_glicemia gl 
  ON f.id_glicemia = gl.id_glicemia
GROUP BY 
    p.faixa_etaria,
    c.descricao_categoria,
    c.classificacao_risco_metabolico,
    gl.faixa_glicemica
ORDER BY 
    pct_prevalencia DESC;